# 発症も死亡もしていない人の `time` は、何年にすればいい？

このノートは、有用性の観点 $U_\text{valid}$（妥当性）の中にある**打ち切り整合（cens）**を扱います（ルールブック §6.1.4）。

**ここで採点する観点**

- 発症も死亡もしていない人の `time` が、追跡期間の終わり（10 年）になっているか。10 年からずれたデータを使う側では、**最後まで追跡された人と、追跡が途中で終わった人の区別が付きません。**

所要 10 分。

## 0. 準備

練習データ（`participant_data/B_practice.csv`）を使います。

In [1]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# 配布データの場所を探す（リポジトリの配置でも、zip を展開しただけの配置でも動くように）
DATA = next(p for p in (Path('..'), Path('../participant_data'), Path('participant_data'))
            if (p / 'B_practice.csv').exists())

from pwscup2026.kit import selfscore as ss        # CodaBench と同じ採点器

B_ref, REF, CFG, info = ss.load_local_reference(DATA)   # 採点器が使う参照値とパラメータ

# 採点と同じ読み方で読む（float_precision='round_trip'）。既定の読み方だと小数の末尾の桁がずれ、
# 値を変えていない列でも点数がごくわずかに下がることがあります。
B = pd.read_csv(DATA / 'B_practice.csv', float_precision='round_trip')

def score(C: pd.DataFrame) -> dict:
    # C を採点して {utility, facets, detail} を返す（CodaBench と同じ採点器）
    return ss.self_score(C.copy(), B_ref, REF, CFG)

print(f'B: {B.shape[0]} 行 × {B.shape[1]} 列')

B: 1049 行 × 14 列


## 1. 転帰の3列と「打ち切り」

データの最後の3列 `time`・`onset`・`death` は、追跡を始めてから何が起きたか（転帰）を表します。

| `onset` | `death` | `time` の意味 |
|---|---|---|
| 1 | 0 | 発症した時点（年） |
| 0 | 1 | 発症する前に亡くなった時点（年） |
| 0 | 0 | 追跡期間の終わり（10 年） |

`onset` と `death` が同時に 1 になることはありません。発症も死亡もしていない人は、10 年の時点で追跡を打ち切ります（ルールブックの「追跡打ち切り時刻 $H$」）。この競技のデータでは、途中で追跡が切れた人はおらず、発症も死亡もしていない人は全員が 10 年間の追跡を終えています（`データはどう作られているか.md` の §3 の作り方のとおりです）。

発症の解析では、発症する前に亡くなった人も「そこまでは発症しなかった」という打ち切りとして扱います（`はじめてのPWSCup.ipynb` の第1節）。このノートで扱うのは、10 年の時点での打ち切りです。

In [2]:
H = float(CFG['onset']['horizon_years'])   # 追跡期間 H（年）
kind = np.select([B.onset == 1, B.death == 1], ['発症した', '発症する前に亡くなった'], '発症も死亡もしていない')
print(f'追跡期間 H = {H} 年')
print('onset と death が両方 1 の行:', int(((B.onset == 1) & (B.death == 1)).sum()))
B.assign(区分=kind).groupby('区分')['time'].agg(人数='size', time_最小='min', time_最大='max')

追跡期間 H = 10.0 年
onset と death が両方 1 の行: 0


,人数,time_最小,time_最大
区分,,,
発症した,110,0.156057,9.946612
発症する前に亡くなった,77,0.024641,9.987680
発症も死亡もしていない,862,9.998631,9.998631


発症も死亡もしていない人の `time` は、全員が同じ値（9.998631）です。`time` は1日単位の日数を 365.25 で割って年に直した値なので、10 年は 9.998631 と表示されます。採点では、H との差が 0.01 年以内なら H と一致しているとみなします。

## 2. 測り方

ルールブック §6.1.4 の打ち切り整合は、次の2つの**小さいほう**です。

- **(a) 行ごとの整合**：発症も死亡もしていない人のうち、`time` が H と一致している人の割合。発症も死亡もしていない人が $C$ に1人もいなければ 0 点。
- **(b) 打ち切り率の一致**：全体のうち `time` が H である人の割合を $p_H$ として、$1-|p_H(C)-p_H(B)|$。

$$\text{cens}=\min\bigl(\text{(a)},\ \text{(b)}\bigr)$$

ルールブックの式どおりに計算して、採点器の値（`censoring_realism_rate`）と一致することを確かめます。

In [3]:
def cens_by_formula(C, B, H, atol=0.01):
    # ルールブック §6.1.4 の打ち切り整合を式どおりに計算する
    near_H = lambda t: np.isclose(np.asarray(t, dtype=float), H, atol=atol)
    censored = (C.onset == 0) & (C.death == 0)                        # 発症も死亡もしていない人
    a = float(near_H(C.loc[censored, 'time']).mean()) if censored.any() else 0.0   # (a)
    b = float(1.0 - min(abs(near_H(C.time).mean() - near_H(B.time).mean()), 1.0))  # (b)
    return {'(a) 行ごとの整合': a, '(b) 打ち切り率の一致': b, 'cens = min': min(a, b)}

print('式どおりに計算:', cens_by_formula(B, B, H))
print('採点器の値    :', score(B)['detail']['U_valid']['censoring_realism_rate'])

式どおりに計算: {'(a) 行ごとの整合': 1.0, '(b) 打ち切り率の一致': 1.0, 'cens = min': 1.0}


採点器の値    : 1.0


## 3. 意味

- **(a)** は、発症も死亡もしていないのに `time` が 10 年でない行、つまり「途中で追跡が切れた人」を作っていないかを見ます。
- **(b)** は、全体の中で 10 年間の追跡を終えた人の割合が、本物と同じかを見ます。
- (a) は、対象（発症も死亡もしていない人）が空なら 0 点です。(b) は全体に対する割合なので、`time` を 10 年から動かした人が多いほど下がります。
- $U_\text{valid}$ は4つの指標（打ち切り整合・関係の非捏造〔県・性別・喫煙による差を本物より強めていないか〕・重複行・行の新規性）の**平均**なので、cens だけで $U_\text{valid}$ が下がる幅は最大 0.25 です。ただし有用性 $U$ は4観点の最小値なので、$U_\text{valid}$ が一番低い観点になれば、そのまま $U$ が下がります。

## 4. 壊してみる

練習データの $B$ に次の2つの加工をして、(a)・(b)・cens がどう動くかを見ます。

1. 全員の `time` に、ランダムな倍率を掛ける（倍率は 1 を中心に正規分布に従ってばらつき、その標準偏差（ばらつき）は 2%）
2. 発症も死亡もしていない人の `time` を 0〜3 年短くする

In [4]:
rng = np.random.default_rng(0)
censored = (B.onset == 0) & (B.death == 0)

C1 = B.copy()      # 1. 全員の time にノイズ
C1['time'] = np.clip(C1.time * (1 + rng.normal(0, 0.02, len(C1))), 0, H)

C2 = B.copy()      # 2. 発症も死亡もしていない人の time を短くする
C2.loc[censored, 'time'] = C2.loc[censored, 'time'] - rng.uniform(0, 3, censored.sum())


rows = {}
for name, C in {'加工なし': B, '1. 全員の time にノイズ': C1,
                '2. 打ち切りの人の time を短く': C2}.items():
    o = score(C)
    rows[name] = {**cens_by_formula(C, B, H), 'U_valid': o['facets']['U_valid'], 'U': o['utility']}
pd.DataFrame(rows).T.round(4)

,(a) 行ごとの整合,(b) 打ち切り率の一致,cens = min,U_valid,U
加工なし,1.0000,1.0000,1.0000,1.0000,1.0000
1. 全員の time にノイズ,0.4872,0.5834,0.4872,0.8718,0.8718
2. 打ち切りの人の time を短く,0.0023,0.1802,0.0023,0.7506,0.7506


### 結果の解釈

- **1.** 発症も死亡もしていない人の `time` も 10 年からずれるので、(a) が約半分に下がります。10 年を超えた分を 10 年で切り詰めたので、ちょうど 10 年になった人が残っています。
- **2.** 発症も死亡もしていない人の `time` がほぼすべて 10 年より短くなるので、(a) はほぼ 0 です。`time` が 10 年の人の割合も 82% からほぼ 0% に減るので、(b) も 0.18 まで下がります。
- $U_\text{valid}$ は4つの平均なので、$U_\text{valid}$ の下がり幅は cens の下がり幅の4分の1です。

### 直し方

`time` を揺らす加工をしたあとで、発症も死亡もしていない人の `time` だけを H に戻すと、打ち切り整合は戻ります。

In [5]:
def keep_censoring(C):
    # 発症も死亡もしていない人の time を H に戻す
    out = C.copy()
    m = (out.onset == 0) & (out.death == 0)
    out.loc[m, 'time'] = H
    return out

rows = {}
for name, C in {'1. 全員の time にノイズ': C1, '1. のあと、打ち切りの人を H に戻す': keep_censoring(C1)}.items():
    o = score(C)
    rows[name] = {**cens_by_formula(C, B, H), 'U_valid': o['facets']['U_valid'], 'U': o['utility']}
pd.DataFrame(rows).T.round(4)

,(a) 行ごとの整合,(b) 打ち切り率の一致,cens = min,U_valid,U
1. 全員の time にノイズ,0.4872,0.5834,0.4872,0.8718,0.8718
1. のあと、打ち切りの人を H に戻す,1.0000,0.9952,0.9952,0.9988,0.9972


(b) がわずかに 1 に届かないのは、発症した人または亡くなった人のうち 5 人が、ノイズで 10 年を超えて 10 年に切り詰められ、`time` が 10 年の人に数えられたためです。

## 5. 自分の C で確かめる

打ち切り整合は、採点結果の `detail` → `U_valid` → `censoring_realism_rate` です。(a) と (b) は別々には出力されないので、どちらで下がっているかは上の `cens_by_formula` で確かめてください。

In [6]:
o = score(C1)
print('censoring_realism_rate =', round(o['detail']['U_valid']['censoring_realism_rate'], 4))
print('式どおりに計算         =', {k: round(v, 4) for k, v in cens_by_formula(C1, B, H).items()})

censoring_realism_rate = 0.4872
式どおりに計算         = {'(a) 行ごとの整合': 0.4872, '(b) 打ち切り率の一致': 0.5834, 'cens = min': 0.4872}


自分の `C.csv` を、`starter/` と `participant_data/` が並んでいるフォルダに置き、そのフォルダで次のコマンドを実行すると、自己採点の `--json` の出力で同じ値を見られます。

```sh
cd starter
uv run python -m pwscup2026.kit.selfscore ../C.csv --dist ../participant_data --json
```

Docker を使う場合:

```sh
docker run --rm -v "$PWD":/w hajimeono/pwscup2026-kit:main-process-20260912 score /w/C.csv --dist /w/participant_data --json
```